In [1]:
import networkx as nx
import pandas as pd
import random
import numpy as np
import math
import json
from pathlib import Path
from unpack_graphs import unpack_graphs
from cup_gather_data import Iteration
from cup_main_faster import updateCR
from cup_functions_faster import update_queue_add_all_neighbors, \
    update_queue_remove_asymmetric, update_queue_remove_unchanged_orbits
    
base_path = Path(r'C:\Users\korol\OneDrive\Pulpit\Master\CUP - remove asymmetric, remove orbits, counting changes\test_graphs')
q_func_names = ["cor", "cas", "cup"]
update_functions = [
    update_queue_remove_unchanged_orbits,
    update_queue_remove_asymmetric,
    update_queue_add_all_neighbors,
]


In [2]:
def nxgraph(n, edges):
    G = nx.Graph()
    G.add_edges_from([tuple(e) for e in edges])
    G.add_nodes_from(range(int(n)))
    return G

def restore_dict_values(hash_dict):
    # hash_dict: {"hash": {"color": int, "orbit": list-or-set}, ...}
    output = {}
    for hash, val in hash_dict.items():
        color = int(val["color"])
        orbit = val.get("orbit")
        if isinstance(orbit, list):  # JSON-serialized set
            orbit = set(orbit)
        output[hash] = {"color": color, "orbit": orbit}
    return output

def unpack(name):
    df = pd.read_csv(base_path / name)
    # dataframe with two columns: G - the graph, iterations with list of iteration objects
    for col in ["graph", "coloring", "hash_dict"]:
        df[col] = df[col].apply(json.loads)
    graphs = []
    iterations = []
    for _, row in df.iterrows():
        graphs.append(nxgraph(row["n"], row["graph"]))
        iter = []
        for coloring_dict, hash_dict_from_json in zip(row["coloring"], row["hash_dict"]):
            coloring = pd.DataFrame(coloring_dict)
            hash_dict = restore_dict_values(hash_dict_from_json)
            iter.append(Iteration(coloring, hash_dict))
        iterations.append(iter)
    return pd.DataFrame({"graph": graphs, "iterations": iterations})
df_all = unpack("random_trees.csv")

In [ ]:
G = df_all["graph"].iloc[0]
it_old = df_all["iterations"].iloc[0]
G_up = G.copy()
G_up.remove_edge(11,79)
print(G_up)
col_cas, _ = updateCR(G_up, {11,79}, it_old, update_queue_remove_asymmetric)
col_cor, _ = updateCR(G_up, {11,79}, it_old, update_queue_remove_unchanged_orbits)

Graph with 100 nodes and 98 edges
h_old = 4.8286414
h_up = 79.9555771
orbit_old = {19}
orbit_up = {19}


In [4]:
print(G_up[19])

{0: {}, 8: {}}


In [4]:
print(q_cas)

[{11, 79}, {0, 48, 11, 79}, {0, 1, 11, 79, 48, 19}, {0, 1, 8, 11, 79, 48, 19, 29}, {0, 11}]


In [5]:
print(q_cor)

[{11, 79}, {0, 48, 11, 79}, {0, 1, 11, 79, 48, 19}, {0, 1, 8, 11, 79, 48, 19, 29}, {0, 11}]


In [6]:
for h, it in it_old[2].hash_dict.items():
    print(f"{h}: {it}")

2.3862943999999997: {'color': 1, 'orbit': {0, 11, 19, 79}}
6.5432948: {'color': 2, 'orbit': {1, 60, 78}}
9.2480429: {'color': 3, 'orbit': {2}}
8.9628447: {'color': 4, 'orbit': {17, 3}}
2.7917595: {'color': 5, 'orbit': {48, 4, 93}}
6.6094379: {'color': 6, 'orbit': {15, 5, 63}}
11.111987800000001: {'color': 7, 'orbit': {6}}
16.8788363: {'color': 8, 'orbit': {7}}
3.6390573: {'color': 9, 'orbit': {8, 37}}
13.2311086: {'color': 10, 'orbit': {9}}
6.9459101: {'color': 11, 'orbit': {10, 76, 46, 86, 62, 95}}
13.4886364: {'color': 12, 'orbit': {12}}
13.574711: {'color': 13, 'orbit': {52, 13}}
14.2046926: {'color': 14, 'orbit': {75, 14}}
14.0775374: {'color': 15, 'orbit': {16, 33}}
7.944439: {'color': 16, 'orbit': {81, 18, 55, 88, 27, 30}}
21.677776899999998: {'color': 17, 'orbit': {20}}
22.968319: {'color': 18, 'orbit': {21}}
23.5847392: {'color': 19, 'orbit': {22}}
22.2456469: {'color': 20, 'orbit': {23}}
12.1506027: {'color': 21, 'orbit': {24}}
7.0910425: {'color': 22, 'orbit': {25, 58, 84}}
4

In [19]:
for h, it in h_cas[-1].items():
    print(f"{h}: {it}")

11.9846527: {'color': 2, 'orbit': {1}}
12.7052195: {'color': 3, 'orbit': {2}}
15.3873736: {'color': 4, 'orbit': {3}}
15.611991199999999: {'color': 5, 'orbit': {4}}
11.6454469: {'color': 6, 'orbit': {5}}
16.9669786: {'color': 7, 'orbit': {6}}
22.7592168: {'color': 8, 'orbit': {7}}
17.7794036: {'color': 9, 'orbit': {8}}
18.908018300000002: {'color': 10, 'orbit': {9}}
17.0426328: {'color': 11, 'orbit': {10}}
17.433722: {'color': 13, 'orbit': {12}}
18.6021188: {'color': 14, 'orbit': {52, 13}}
19.0937502: {'color': 15, 'orbit': {14}}
16.6094379: {'color': 16, 'orbit': {15}}
20.8751973: {'color': 17, 'orbit': {16, 33}}
26.7513266: {'color': 18, 'orbit': {17}}
24.0707377: {'color': 19, 'orbit': {18}}
22.8286414: {'color': 20, 'orbit': {19}}
36.596579: {'color': 21, 'orbit': {20}}
36.4159446: {'color': 22, 'orbit': {21}}
38.6694155: {'color': 23, 'orbit': {22}}
33.3749588: {'color': 24, 'orbit': {23}}
33.9268622: {'color': 25, 'orbit': {24}}
33.759511700000004: {'color': 26, 'orbit': {25}}
34.

In [20]:
for h, it in h_cor[-1].items():
    print(f"{h}: {it}")

11.9846527: {'color': 2, 'orbit': {1}}
12.7052195: {'color': 3, 'orbit': {2}}
15.3873736: {'color': 4, 'orbit': {3}}
15.611991199999999: {'color': 5, 'orbit': {4}}
11.6454469: {'color': 6, 'orbit': {5}}
16.9669786: {'color': 7, 'orbit': {6}}
22.7592168: {'color': 8, 'orbit': {7}}
17.7794036: {'color': 9, 'orbit': {8}}
18.908018300000002: {'color': 10, 'orbit': {9}}
17.0426328: {'color': 11, 'orbit': {10}}
17.433722: {'color': 13, 'orbit': {12}}
18.6021188: {'color': 14, 'orbit': {52, 13}}
19.0937502: {'color': 15, 'orbit': {14}}
16.6094379: {'color': 16, 'orbit': {15}}
20.8751973: {'color': 17, 'orbit': {16, 33}}
26.7513266: {'color': 18, 'orbit': {17}}
24.0707377: {'color': 19, 'orbit': {18}}
36.596579: {'color': 21, 'orbit': {20}}
36.4159446: {'color': 22, 'orbit': {21}}
38.6694155: {'color': 23, 'orbit': {22}}
33.3749588: {'color': 24, 'orbit': {23}}
33.9268622: {'color': 25, 'orbit': {24}}
33.759511700000004: {'color': 26, 'orbit': {25}}
34.9267831: {'color': 27, 'orbit': {26}}
32.